# GPU Experiment 2: Long Generation (max_new_tokens=800)
This notebook proves that the metrics hold up when outputs naturally terminate (EOS Hit > 90%), addressing Reviewer's truncation concern.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes evaluate bert_score

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import numpy as np
from evaluate import load
from tqdm import tqdm
import time

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
)
bertscore = load("bertscore")

In [ ]:
# Load previously extracted steering vector using dynamic path search
import glob
matches = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True)
if not matches:
    matches = glob.glob('/kaggle/input/**/*.pt', recursive=True)

if not matches:
    raise FileNotFoundError("Could not find v_steer.pt in /kaggle/input. Please ensure the dataset is attached to the notebook!")

vec_path = matches[0]
print(f"Found vector file at: {vec_path}")

vec_data = torch.load(vec_path, map_location='cpu')
if isinstance(vec_data, dict):
    v_steer = vec_data.get('steering_vector', vec_data.get('v_steer', list(vec_data.values())[0]))
else:
    v_steer = vec_data
v_steer = v_steer.to(model.device).to(torch.float16)
print("Loaded vector successfully!")

# Load Test Data using dynamic glob search
json_files = glob.glob('/kaggle/input/**/vietnamese_medical_halueval_15k_specialized.json', recursive=True)
if not json_files:
    json_files = glob.glob('/kaggle/input/**/*.json', recursive=True)

with open(json_files[0], 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"Loaded dataset from: {json_files[0]}")
test_data = data[-500:]

def format_prompt(q):
    messages = [{"role": "system", "content": "You are a helpful and accurate medical assistant."}, {"role": "user", "content": q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
def evaluate_long(use_steering=False, alpha=18.0, K=16):
    hook_handle = None
    step_counter = [0]
    if use_steering:
        def steering_hook(module, input, output):
            step_counter[0] += 1
            t = step_counter[0]
            if t <= K:
                alpha_t = alpha * (1.0 - (t - 1) / K)
                if isinstance(output, tuple):
                    h = output[0]
                    v = v_steer.to(device=h.device, dtype=h.dtype)
                    h[:, -1, :] += alpha_t * v
                    return (h,) + output[1:]
                else:
                    v = v_steer.to(device=output.device, dtype=output.dtype)
                    output[:, -1, :] += alpha_t * v
                    return output
            return output
        hook_handle = model.model.layers[8].register_forward_hook(steering_hook)
        
    generated, refs, hals, eos_hits = [], [], [], []
    
    for item in tqdm(test_data, desc="Evaluating"):
        step_counter[0] = 0
        prompt = format_prompt(item['question'])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_length = inputs.input_ids.shape[1]
        
        # KEY CHANGE: max_new_tokens=800
        out = model.generate(**inputs, max_new_tokens=800, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        
        gen_tokens = out[0][input_length:]
        eos_hits.append(tokenizer.eos_token_id in gen_tokens or tokenizer.pad_token_id in gen_tokens)
        
        generated.append(tokenizer.decode(gen_tokens, skip_special_tokens=True))
        refs.append(item.get('right_answer', item.get('positive_answer')))
        hals.append(item['hallucinated_answer'])
        
    if hook_handle: hook_handle.remove()
    
    # Compute BERTScore
    bs_ref = bertscore.compute(predictions=generated, references=refs, model_type="bert-base-multilingual-cased")['f1']
    bs_hal = bertscore.compute(predictions=generated, references=hals, model_type="bert-base-multilingual-cased")['f1']
    
    correct = sum(1 for r, h in zip(bs_ref, bs_hal) if r > h)
    
    return {
        "correct": correct,
        "total": len(test_data),
        "accuracy": correct / len(test_data) * 100,
        "eos_hit_rate": sum(eos_hits) / len(eos_hits) * 100,
        "mean_bs": np.mean(bs_ref)
    }

print("Running Baseline (800 tokens)...")
base_res = evaluate_long(use_steering=False)

print("Running Steered (800 tokens)...")
steer_res = evaluate_long(use_steering=True)

print("\n=== LONG GENERATION RESULTS ===")
print(f"Baseline: Acc={base_res['accuracy']:.2f}%, EOS Hit={base_res['eos_hit_rate']:.2f}%, BS={base_res['mean_bs']:.4f}")
print(f"Steered : Acc={steer_res['accuracy']:.2f}%, EOS Hit={steer_res['eos_hit_rate']:.2f}%, BS={steer_res['mean_bs']:.4f}")
